# ASTR 457: Foundations of Data Science in Astronomy

**Fall 2026, University of Illinois Urbana-Champaign**

Prof. Gautham Narayan | TA: Abha Vishwakarma

Thu Sep 24, 2026 - Day 10 - Regression II: errors in both variables

<img src="images/uiuc_logo.png" alt="University of Illinois Urbana-Champaign wordmark" width="220" style="display:block;margin:0 auto;">

## Before we start: a one-time fix

In a terminal (WSL on Windows):

```
conda activate astr457
conda install -c conda-forge "ipykernel=6.31"
```

Then quit and restart Jupyter, and reopen today's notebook.

* today's notebook has sliders, and with ipykernel 7 its plot labels can fail (`ParseException: Expected end of text`) or plots can silently not appear
* version 6.31 does not have the problem
* the same step is now in `INSTALL.txt`

## Today's plan

* correlated measurement errors, then measurement errors in $x$
* transit light curves, and black holes in dwarf galaxies
* the generative model, from Charlotte Ward's talk
* your lab error bars vs. the truth, Lab 04 and Charbonneau

In [ ]:
# Presenter setup (a RISE "skip" cell: it runs, but is not a slide).
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
from scipy import stats, odr

# matplotlib's math-text parser is a shared singleton and is not thread-safe. ipykernel 7 handles slider messages on
# another thread (ipykernel #1564), so a redraw can overlap the next cell and break any "$...$" label
# ("Expected end of text", ipympl #610). A lock around the parser fixes that on any kernel version (ipympl PR #621).
import threading, functools
try:
    from matplotlib import _mathtext
    if not getattr(_mathtext.Parser.parse, '_locked', False):
        _mathtext_lock = threading.Lock()
        _unlocked_parse = _mathtext.Parser.parse
        def _locked_parse(self, *args, **kwargs):
            with _mathtext_lock:
                return _unlocked_parse(self, *args, **kwargs)
        _locked_parse._locked = True
        _mathtext.Parser.parse = _locked_parse
except (ImportError, AttributeError):
    print("could not lock matplotlib's math-text parser: do not move a slider while another cell is running")

import ipykernel
if int(ipykernel.__version__.split('.')[0]) >= 7:
    print(f"ipykernel {ipykernel.__version__}: this notebook is tested on 6.31 (see the first slide); labels are protected by the lock above")

_PLOT_LOCK = threading.RLock()

def one_plot_at_a_time(fn):
    # pyplot's "current figure" is global too, so whole plot functions take turns
    @functools.wraps(fn)
    def wrapped(*args, **kwargs):
        with _PLOT_LOCK:
            return fn(*args, **kwargs)
    return wrapped

def load_csv(path):
    with open(path) as _f:
        _nhead = sum(1 for _line in _f if _line.startswith('#'))
    return np.genfromtxt(path, delimiter=',', skip_header=_nhead, names=True, dtype=None, encoding='utf-8')

def ols(A, y):
    """theta_hat, residual-scaled covariance, residuals (Tuesday's normal equations)."""
    AtA_inv = np.linalg.inv(A.T @ A)
    theta = AtA_inv @ A.T @ y
    resid = y - A @ theta
    return theta, (resid @ resid / (len(y) - A.shape[1])) * AtA_inv, resid

def gls(A, y, Sigma):
    """theta_hat and covariance for a full covariance matrix Sigma (WLS when Sigma is diagonal)."""
    Si = np.linalg.inv(Sigma)
    cov = np.linalg.inv(A.T @ Si @ A)
    return cov @ A.T @ Si @ y, cov

# Hubble 1929 Table 1 again, for the recap plot (provenance in the CSV header)
hub = load_csv('data/hubble1929_table1.csv')
r_hub, v_hub = hub['r_Mpc'].astype(float), hub['v_kms'].astype(float)

# The demo relation: a SIMULATED dwarf-AGN sample with a planted truth, revealed on the slide where it matters.
# Same structure as Lab 04 (x = log stellar mass, y = log black-hole mass, both with errors, intrinsic scatter), different seed and truth.
TRUE = dict(alpha=1.35, beta=6.25, sig_int=0.40, mu=9.5, sig_pop=0.60)
rng_demo = np.random.default_rng(20260922)
N_demo = 40
x_true = rng_demo.normal(TRUE['mu'], TRUE['sig_pop'], N_demo)
y_true = TRUE['alpha'] * (x_true - 9.5) + TRUE['beta'] + rng_demo.normal(0, TRUE['sig_int'], N_demo)
sx = rng_demo.uniform(0.2, 0.4, N_demo); sy = rng_demo.uniform(0.25, 0.4, N_demo)
x_obs = x_true + rng_demo.normal(0, sx); y_obs = y_true + rng_demo.normal(0, sy)
print(f"demo sample: {N_demo} dwarf AGN, planted alpha = {TRUE['alpha']}, median sigma_x = {np.median(sx):.2f} dex, sig_pop = {TRUE['sig_pop']}")

## Recap: Hubble's two slopes

* in astrophysics the errors are seldom in one variable only, and seldom Gaussian
* least squares assumes three things:
  * Gaussian noise (Tuesday's outliers broke this)
  * exact $x$
  * independent points
* the same 24 galaxies, fit two ways
* which line would you trust, and which way is his slope biased?
* Tuesday ended with: write down how the data were made - today we write one down

In [ ]:
# Visual reminder: Tuesday's two least-squares lines through Hubble's 24 nebulae (a "skip" cell)
A_vr = np.vstack([np.ones_like(r_hub), r_hub]).T
th_vr, _, _ = ols(A_vr, v_hub)
A_rv = np.vstack([np.ones_like(v_hub), v_hub]).T
th_rv, _, _ = ols(A_rv, r_hub)
K_inv, r0_inv = 1.0 / th_rv[1], th_rv[0]          # r = r0 + v / K'  ->  v = K' (r - r0)

@one_plot_at_a_time
def plot_recap():
    fig, ax = plt.subplots(figsize=(6, 2.8))
    rr = np.linspace(-0.1, 2.1, 20)
    ax.plot(r_hub, v_hub, 'o', color='C0', ms=5)
    ax.plot(rr, th_vr[0] + th_vr[1] * rr, 'C3', lw=1.6, label=fr'$v$ on $r$: $K$ = {th_vr[1]:.0f}')
    ax.plot(rr, K_inv * (rr - r0_inv), 'C2', lw=1.6, label=fr'$r$ on $v$: $K$ = {K_inv:.0f}')
    ax.set_xlabel('$r$ [Mpc]'); ax.set_ylabel('$v$ [km/s]'); ax.legend(frameon=False, fontsize=9)
    plt.tight_layout()
    plt.show()

In [ ]:
# Tuesday's two slopes from one dataset: each assumes the errors are in a different variable
plot_recap()

## Correlated measurement errors

* first, the assumption whose failure changes only the error bar: independent points

<img src="images/pont2006_fig2_white_red_noise.png" alt="Pont, Zucker and Queloz 2006 Figure 2: three simulated light curves, white noise only, red noise only, and both" style="display:block;margin:0 auto;max-height:190px">

<div style="font-size:0.8em; text-align:center; color:#666;">Pont, Zucker &amp; Queloz 2006, MNRAS 373, 231, Fig. 2: simulated transit light curves with white noise (top), red noise (middle), both (bottom).</div>

* transit photometry from the ground: seeing, weather and the detector drift over hours, so each point's error is tied to its neighbor's - "red noise"
* adjacent points have correlation $\rho$ ($\rho^k$ at $k$ apart); here $\rho = 0.9$


## Generalized least squares

* generalized least squares (GLS) is Tuesday's normal equations with the full $\Sigma$ instead of a diagonal one:

$$\hat{\boldsymbol\theta} = (A^{\mathsf T}\Sigma^{-1}A)^{-1}A^{\mathsf T}\Sigma^{-1}\mathbf{y}, \qquad \mathrm{Cov}(\hat{\boldsymbol\theta}) = (A^{\mathsf T}\Sigma^{-1}A)^{-1}$$

* red plus white noise: $\Sigma_{ij} = \sigma_r^2\,\rho^{|i-j|} + \sigma_w^2\,\delta_{ij}$, the matrix drawn on the next slides


In [ ]:
from ipywidgets import interact, FloatSlider, IntSlider, fixed
# A straight line under correlated (AR(1)) noise: OLS and GLS on one realization, then 500 (a "skip" cell)
rng_gls = np.random.default_rng(20260924)
t = np.linspace(0, 10, 60)
a_true, b_true = 1.0, 0.3
sig_w, sig_r, rho = 0.10, 0.25, 0.9              # white noise, red-noise amplitude, correlation between neighbours
lag = np.abs(t[:, None] - t[None, :]) / (t[1] - t[0])
Sigma_red = sig_r**2 * rho**lag + sig_w**2 * np.eye(t.size)
L_red = np.linalg.cholesky(Sigma_red)
A_t = np.vstack([np.ones_like(t), t]).T

def one_realization():
    return a_true + b_true * t + L_red @ rng_gls.normal(size=t.size)

y_one = one_realization()
th_o, cov_o, _ = ols(A_t, y_one)
th_g, cov_g = gls(A_t, y_one, Sigma_red)

z_ols, z_gls, th_ols_all = [], [], []
for _ in range(500):
    y_k = one_realization()
    to, co, _ = ols(A_t, y_k); tg, cg = gls(A_t, y_k, Sigma_red)
    th_ols_all.append(to)
    z_ols.append((to[1] - b_true) / np.sqrt(co[1, 1])); z_gls.append((tg[1] - b_true) / np.sqrt(cg[1, 1]))
z_ols, z_gls, th_ols_all = np.array(z_ols), np.array(z_gls), np.array(th_ols_all)

@one_plot_at_a_time
def plot_gls(n_lines=50):
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.8), gridspec_kw=dict(width_ratios=[0.9, 1.7, 1.2]))
    ax = axes[0]
    ax.imshow(Sigma_red, cmap='magma'); ax.set_title(fr'$\Sigma$, $\rho$ = {rho}', fontsize=10); ax.set_xticks([]); ax.set_yticks([]); ax.set_xlabel('point $j$'); ax.set_ylabel('point $i$')
    ax = axes[1]
    tm = t.mean()
    band_hi = th_o[0] + th_o[1] * tm + (th_o[1] + np.sqrt(cov_o[1, 1])) * (t - tm)
    band_lo = th_o[0] + th_o[1] * tm + (th_o[1] - np.sqrt(cov_o[1, 1])) * (t - tm)
    for th in th_ols_all[:n_lines]:
        ax.plot(t, th[0] + th[1] * t, color='C2', lw=0.6, alpha=0.35)
    ax.fill_between(t, band_lo, band_hi, color='C1', alpha=0.55, zorder=4, label="one fit's quoted OLS $\\pm 1\\sigma$ slope")
    ax.plot(t, band_lo, color='C1', lw=1.2, zorder=5); ax.plot(t, band_hi, color='C1', lw=1.2, zorder=5)
    ax.plot(t, a_true + b_true * t, 'k', ls='--', lw=1.6, zorder=6, label='truth')
    ax.plot([], [], color='C2', lw=1, label=f'OLS fits to {n_lines} noise draws')
    ax.set_xlabel('$t$'); ax.set_ylim(-1, 6); ax.legend(frameon=False, fontsize=9, loc='upper left')
    ax = axes[2]
    bins = np.linspace(min(z_ols.min(), -3), max(z_ols.max(), 3), 41)
    ax.hist(z_ols, bins, histtype='step', color='C2', lw=1.6, label=f'OLS, std {z_ols.std():.1f}')
    ax.hist(z_gls, bins, histtype='step', color='C3', lw=1.6, label=f'GLS, std {z_gls.std():.1f}')
    ax.axvspan(-2, 2, color='0.9', zorder=0)
    ax.set_xlabel(r'(slope $-$ truth) / quoted $\sigma$'); ax.set_title('all 500 draws', fontsize=10); ax.legend(frameon=False, fontsize=9)
    plt.tight_layout()
    plt.show()
    print(f"OLS quoted sigma_slope is too small by a factor {z_ols.std():.1f}; {np.mean(np.abs(z_ols) > 2):.0%} of OLS fits are more than 2 quoted sigma off; GLS z std {z_gls.std():.2f}")


In [ ]:
# Three red-noise draws from the same Sigma, next to one white-noise draw (a "skip" cell)
@one_plot_at_a_time
def plot_red_draws():
    rng_d = np.random.default_rng(11)
    fig, ax = plt.subplots(figsize=(9, 3.2))
    ax.plot(t, sig_w * rng_d.normal(size=t.size) + 0.9, 'o-', color='0.6', ms=3, lw=0.8, label='white noise (diagonal $\\Sigma$)')
    for k, col in enumerate(['C0', 'C1', 'C4']):
        ax.plot(t, L_red @ rng_d.normal(size=t.size) - 0.6 * k, 'o-', color=col, ms=3, lw=0.8, label=f'red noise, draw {k + 1}' if k == 0 else f'draw {k + 1}')
    ax.set_xlabel('$t$'); ax.set_ylabel('noise (offset for clarity)'); ax.set_yticks([])
    ax.legend(frameon=False, fontsize=8, ncol=4, loc='lower center', bbox_to_anchor=(0.5, 1.0))
    plt.tight_layout()
    plt.show()

## Drawing red noise from $\Sigma$

* each draw is $L\mathbf{z}$, with $\mathbf{z}$ standard normal and $LL^{\mathsf T} = \Sigma$ (a Cholesky factor)
* neighbors move together, so a draw wanders instead of jittering
* the next slide fits a line to 500 draws like these

In [ ]:
# gray: white noise; colors: three draws from the same red-noise Sigma
plot_red_draws()

## 500 OLS fits to red noise

* each green line is OLS on a new draw of the same red noise
* the orange band is one fit's quoted $\pm 1\sigma$ on the slope
* try adding lines to see how many stay inside the band

In [ ]:
# left: the covariance matrix; middle: OLS fits to many noise draws against one fit's quoted error; right: every fit's error checked against the truth
interact(plot_gls, n_lines=IntSlider(value=50, min=1, max=500, step=1));

* all 500 OLS error bars are 4$\times$ too small, even for fits near the truth
* 4$\times$ in $\sigma$ is 16$\times$ in $N$: 60 correlated points are worth ~4 independent ones
* GLS gets the error bar right
* (red noise in real data, and fitting $\Sigma$ with Gaussian processes: the time-series lecture, Oct 20/22)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**With correlated noise, all 500 OLS error bars are too small. The diagonal $\Sigma$ is the wrong model!**

</div>

## Measurement errors in $x$

* back to Hubble: which line would you trust, and which way is his slope biased?
* measurement errors in $x$ bias the slope itself (red noise only got the error bar wrong)
* $\sigma_{\rm pop}$ is the standard deviation of the true $x$ values - how much the galaxies really differ
* $\sigma_x$ is the measurement error on each $x$

<img src="images/kelly2007_fig3_noise_levels.png" alt="Kelly 2007 Figure 3: one simulated sample of true values, then the same sample observed with increasing measurement error in x and y, the box marking the true-value range" style="display:block;margin:0 auto;max-height:280px">

<div style="font-size:0.8em; text-align:center; color:#666;">Kelly 2007, ApJ 665, 1489, Fig. 3: true values $\xi$, $\eta$ (top left), then the same sample with more and more measurement error ($R_x$ = 0.2, 0.5, 0.8, defined next).</div>

In [ ]:
# Schematic: true points on the line, the same points after x noise, and the least-squares line through the noisy ones (a "skip" cell)
@one_plot_at_a_time
def plot_noise_in_x():
    rng_s = np.random.default_rng(3)
    xt = np.sort(rng_s.uniform(-1, 1, 14)); yt = 1.0 * xt
    xo = xt + rng_s.normal(0, 0.45, xt.size)
    A_s = np.vstack([np.ones_like(xo), xo]).T
    th_s = np.linalg.inv(A_s.T @ A_s) @ A_s.T @ yt
    fig, ax = plt.subplots(figsize=(7, 3.6))
    xx = np.linspace(-1.8, 1.8, 10)
    ax.plot(xx, xx, '0.4', ls='--', lw=1.4, label='the true relation, slope 1')
    ax.plot(xt, yt, 'o', color='0.6', ms=5, label='true $x$')
    for a, b, y in zip(xt, xo, yt):
        ax.annotate('', xy=(b, y), xytext=(a, y), arrowprops=dict(arrowstyle='->', color='C0', lw=0.9))
    ax.plot(xo, yt, 'o', color='C0', ms=5, label='observed $x$ (noise added sideways)')
    ax.plot(xx, th_s[0] + th_s[1] * xx, 'C2', lw=2, label=f'least squares on the observed $x$: slope {th_s[1]:.2f}')
    ax.set_xlabel('$x$'); ax.set_ylabel('$y$'); ax.legend(frameon=False, fontsize=8, loc='upper left')
    plt.tight_layout()
    plt.show()

## Errors in $x$ and the fit

* let's put points exactly on a line, scatter each one left and right, and refit

In [ ]:
# gray: points on the true line; blue: the same points scattered in x; green: the refit
plot_noise_in_x()

* the points scatter left and right
* the best-fit line is flatter than the true relation
* averaged over many samples (its expectation value $E[\,]$), the OLS slope $\hat\alpha_{\textrm{OLS}}$ is

$$E[\hat\alpha_{\textrm{OLS}}] = \alpha\,\frac{\sigma_{\rm pop}^2}{\sigma_{\rm pop}^2 + \sigma_x^2} = \alpha\,(1 - R_x)$$

* $R_x = \sigma_x^2/(\sigma_{\rm pop}^2 + \sigma_x^2)$ is the fraction of the observed $x$ variance that is noise (Kelly's notation)
* always toward zero, and called **attenuation** (or regression dilution)
* so Hubble's $K$ was biased low
* his factor-of-7 calibration error is separate: it rescales every distance ($K$ too high), and no fit can find it

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Errors in $x$ bias the OLS slope low. More data does not help!**

</div>

In [ ]:
# Widget: OLS slope against the attenuation prediction as sigma_x, sig_pop and N change (a "skip" cell)
@one_plot_at_a_time
def attn_widget(sig_x=0.3, sig_pop=0.6, N=40):
    rng_a = np.random.default_rng(0)
    xt = rng_a.normal(0, sig_pop, N)
    yv = 1.35 * xt + rng_a.normal(0, 0.3, N)
    xo = xt + rng_a.normal(0, sig_x, N)
    b = np.polyfit(xo, yv, 1)[0]
    pred = 1.35 * sig_pop**2 / (sig_pop**2 + sig_x**2)
    fig, ax = plt.subplots(figsize=(7, 3.6))
    xx = np.linspace(-2.5, 2.5, 5)
    ax.plot(xo, yv, 'o', color='C0', ms=3, alpha=0.6)
    ax.plot(xx, 1.35 * xx, '0.4', ls='--', lw=1.4, label='truth: 1.35')
    ax.plot(xx, b * xx, 'C2', lw=2, label=f'OLS: {b:.2f}')
    ax.plot(xx, pred * xx, 'k', ls=':', lw=1.4, label=f'predicted: {pred:.2f}')
    ax.set_xlim(-2.5, 2.5); ax.set_ylim(-4, 4); ax.set_xlabel('$x$, observed'); ax.set_ylabel('$y$')
    ax.legend(frameon=False, fontsize=9, loc='upper left')
    plt.tight_layout()
    plt.show()

## Try it: attenuation

* try changing $\sigma_x$ to see the OLS slope fall toward zero
* a bigger $N$ does not bring it back

In [ ]:
# sigma_x, sig_pop and N; the dotted line is sig_pop^2 / (sig_pop^2 + sigma_x^2) times the truth
interact(attn_widget, sig_x=FloatSlider(value=0.3, min=0.0, max=1.2, step=0.05), sig_pop=FloatSlider(value=0.6, min=0.2, max=1.5, step=0.05), N=IntSlider(value=40, min=10, max=3000, step=10));

## A relation with errors on both axes

<div style="display:flex; justify-content:center; align-items:flex-start; gap:1em;">
<img src="images/henize2-10_hst_opo2202a.jpg" alt="Hubble image of the dwarf starburst galaxy Henize 2-10, which hosts a central massive black hole" style="max-height:290px;">
<img src="images/reines2015_mbh_mstar.png" alt="Reines and Volonteri 2015 Figure 8 left: black-hole mass against host stellar mass for broad-line AGN, dwarf AGN and dynamical black holes, four published relations drawn as lines" style="max-height:290px;">
</div>

<div style="font-size:0.8em; text-align:center; color:#666;">Left: Henize 2-10, a dwarf galaxy with a central black hole. NASA, ESA, Z. Schutte (XGI), A. Reines (XGI), A. Pagan (STScI); CC BY 4.0. Right: Reines &amp; Volonteri 2015, ApJ 813, 82 (arXiv:1508.06274), Fig. 8 (left): $M_{\rm BH}$ against $M_*$ for 244 broad-line AGN, reverberation-mapped AGN, dwarf AGN and dynamical masses, with four published relations.</div>

* black-hole mass tracks host stellar mass, but below $M_* \approx 10^{9.5}\,M_\odot$ it is poorly measured
* dwarfs like Henize 2-10 grew little, and test how the first black holes ("seeds") formed
* **both axes are measurements**
* $M_{\textrm{BH}}$ from line width and luminosity, ~0.5 dex (1 dex is a factor of 10)
* $M_*$ from photometry, ~0.3 dex
* intrinsic scatter of 0.24 dex for the AGN relation, fit with Kelly 2007's method

In [ ]:
# The demo relation: OLS on the observed points, against the planted truth, then 1000 fake samples (a "skip" cell)
A_d = np.vstack([np.ones(N_demo), x_obs - 9.5]).T
th_d, cov_d, _ = ols(A_d, y_obs)
th_dw, cov_dw = gls(A_d, y_obs, np.diag(sy**2))          # WLS with the y errors only
predicted = TRUE['alpha'] * TRUE['sig_pop']**2 / (TRUE['sig_pop']**2 + np.median(sx)**2)

rng_mc = np.random.default_rng(1)
mc_slopes = []
for _ in range(1000):
    xt = rng_mc.normal(TRUE['mu'], TRUE['sig_pop'], N_demo)
    yt = TRUE['alpha'] * (xt - 9.5) + TRUE['beta'] + rng_mc.normal(0, TRUE['sig_int'], N_demo)
    xo = xt + rng_mc.normal(0, sx); yo = yt + rng_mc.normal(0, sy)
    mc_slopes.append(ols(np.vstack([np.ones(N_demo), xo - 9.5]).T, yo)[0][1])
mc_slopes = np.array(mc_slopes)

@one_plot_at_a_time
def plot_attenuation():
    fig, (ax, ah) = plt.subplots(1, 2, figsize=(10, 4.0), gridspec_kw=dict(width_ratios=[1.5, 1]))
    xx = np.linspace(7.8, 11.2, 20)
    ax.errorbar(x_obs, y_obs, xerr=sx, yerr=sy, fmt='o', color='C0', ms=4, elinewidth=0.7, capsize=0, label='simulated dwarf AGN')
    ax.plot(xx, TRUE['beta'] + TRUE['alpha'] * (xx - 9.5), '0.4', ls='--', lw=1.4, label=fr"truth: $\alpha$ = {TRUE['alpha']:.2f}, $\sigma_\mathrm{{int}}$ = {TRUE['sig_int']:.2f}")
    ax.plot(xx, th_d[0] + th_d[1] * (xx - 9.5), 'C2', lw=1.8, label=fr'OLS: $\alpha$ = {th_d[1]:.2f} $\pm$ {np.sqrt(cov_d[1, 1]):.2f}')
    ax.plot(xx, th_dw[0] + th_dw[1] * (xx - 9.5), 'C1', lw=1.4, ls=':', label=fr'WLS ($y$ errors): $\alpha$ = {th_dw[1]:.2f} $\pm$ {np.sqrt(cov_dw[1, 1]):.2f}')
    ax.set_xlabel(r'$\log_{10}(M_*/M_\odot)$, observed'); ax.set_ylabel(r'$\log_{10}(M_{\rm BH}/M_\odot)$, observed')
    ax.legend(frameon=False, fontsize=9)
    ah.hist(mc_slopes, 40, color='C2', alpha=0.7)
    ah.axvline(TRUE['alpha'], color='0.3', ls='--', lw=1.4, label='truth')
    ah.axvline(mc_slopes.mean(), color='C2', lw=1.6, label=f'mean OLS slope {mc_slopes.mean():.2f}')
    ah.axvline(predicted, color='k', lw=1.0, ls=':', label=f'predicted {predicted:.2f}')
    ah.set_xlabel(r'OLS $\hat\alpha$ over 1000 fake samples'); ah.legend(frameon=False, fontsize=9)
    plt.tight_layout()
    plt.show()
    print(f"planted alpha = {TRUE['alpha']}; OLS on this sample {th_d[1]:.2f} +/- {np.sqrt(cov_d[1, 1]):.2f}; WLS {th_dw[1]:.2f}; "
          f"mean over 1000 samples {mc_slopes.mean():.2f} (predicted {predicted:.2f}, {100*(1-predicted/TRUE['alpha']):.0f}% low)")

## Demo: a simulated dwarf-AGN sample

* the real relation has no known truth, so no fit can be graded on it - we simulate one with a planted answer (Lab 04 does the same)
* 40 galaxies, $x = \log M_*$ and $y = \log M_{\textrm{BH}}$, both measured with errors
* the model: $y_{\rm true} = \alpha\,(x_{\rm true} - 9.5) + \beta + \mathcal{N}(0, \sigma_{\textrm{int}}^2)$
* planted: $\alpha = 1.35$, $\beta = 6.25$, $\sigma_{\textrm{int}} = 0.40$, $\sigma_{\rm pop} = 0.60$, $\sigma_x \approx 0.30$
* so $\sigma_x/\sigma_{\rm pop} \approx 0.5$, $R_x = 0.2$, and OLS should average $1.35 \times 0.8 = 1.08$


In [ ]:
# OLS on the real Reines & Volonteri 2015 sample, for the read-off above (a "skip" cell)
rv = load_csv('data/reines2015_table1.csv')
x_rv, y_rv = rv['log_Mstar'].astype(float), rv['log_MBH'].astype(float)
th_rv15, cov_rv15, _ = ols(np.vstack([np.ones_like(x_rv), x_rv - 9.5]).T, y_rv)
print(f"Reines & Volonteri 2015 Table 1: {x_rv.size} AGN; OLS slope {th_rv15[1]:.2f} +/- {np.sqrt(cov_rv15[1, 1]):.2f}; their Kelly 2007 fit: 1.05 +/- 0.11")


## OLS on the simulated sample

* one sample (left) and OLS on 1000 samples (right)


In [ ]:
# left: one simulated sample with the truth and two least-squares lines; right: the OLS slope over a thousand such samples
plot_attenuation()

* 1000 samples average 1.08, the predicted bias
* this sample (0.90 $\pm$ 0.15) sits low in the histogram - a bit unlucky
* weighting by the $y$ errors (WLS, 0.97) can't fix errors in $x$
* on the real 244 AGN (Reines & Volonteri Table 1): OLS gives 0.55 $\pm$ 0.07, their Kelly fit 1.05 $\pm$ 0.11

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**$\sigma_x/\sigma_{\rm pop}$ predicts the bias before you fit anything.**

</div>

## Orthogonal distance regression

* least squares measures each point's distance vertically, because only $y$ had noise
* orthogonal distance regression (ODR) measures it perpendicular to the line, scaled by both error bars
* ODR assumes no intrinsic scatter
* Tuesday's bisector just averages two biased lines
* try changing the slope below to see where each sum is smallest

In [ ]:
# Schematic: vertical misses (least squares) against perpendicular misses (ODR) for the same points (a "skip" cell)
@one_plot_at_a_time
def plot_distances(a=0.8):
    xp = np.linspace(-1.5, 1.5, 7); yp = 0.8 * xp + np.array([0.55, -0.6, 0.35, -0.5, 0.65, -0.4, 0.45])
    b = 0.0
    vert = np.sum((yp - a * xp)**2)
    perp = vert / (1 + a**2)
    fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.6), sharey=False)
    xx = np.linspace(-2, 2, 10)
    for ax, kind in zip(axes, ['vertical', 'perpendicular']):
        ax.plot(xx, a * xx + b, 'k', lw=1.4)
        for x0, y0 in zip(xp, yp):
            if kind == 'vertical':
                xf, yf = x0, a * x0 + b
            else:
                xf = (x0 + a * (y0 - b)) / (1 + a**2); yf = a * xf + b
            ax.plot([x0, xf], [y0, yf], color='C3' if kind == 'vertical' else 'C4', lw=1.6)
        ax.plot(xp, yp, 'o', color='C0', ms=6, zorder=3)
        ax.set_xlim(-2, 2); ax.set_ylim(-2.5, 2.5); ax.set_aspect('equal', adjustable='box'); ax.set_xlabel('$x$')
        ax.set_title(f'vertical: sum of squares {vert:.2f}' if kind == 'vertical' else f'perpendicular: sum of squares {perp:.2f}', fontsize=10)
    axes[0].set_ylabel('$y$')
    plt.tight_layout()
    plt.show()

In [ ]:
# least squares minimizes the left sum, ODR the right one
interact(plot_distances, a=FloatSlider(value=0.8, min=0.0, max=2.5, step=0.05));

In [ ]:
# ODR on the demo sample, with both error bars (a "skip" cell)
def _line(B, x):
    return B[0] * (x - 9.5) + B[1]
_odr = odr.ODR(odr.RealData(x_obs, y_obs, sx=sx, sy=sy), odr.Model(_line), beta0=[1.0, 6.3])
odr_out = _odr.run()
alpha_odr, beta_odr = odr_out.beta
sig_alpha_odr = odr_out.sd_beta[0]


rng_o = np.random.default_rng(2)
odr_slopes = []
for _ in range(1000):
    xt = rng_o.normal(TRUE['mu'], TRUE['sig_pop'], N_demo)
    yt = TRUE['alpha'] * (xt - 9.5) + TRUE['beta'] + rng_o.normal(0, TRUE['sig_int'], N_demo)
    xo = xt + rng_o.normal(0, sx); yo = yt + rng_o.normal(0, sy)
    odr_slopes.append(odr.ODR(odr.RealData(xo, yo, sx=sx, sy=sy), odr.Model(_line), beta0=[1.0, 6.3]).run().beta[0])
odr_slopes = np.array(odr_slopes)


@one_plot_at_a_time
def plot_odr():
    fig, (ax, ah) = plt.subplots(1, 2, figsize=(10, 4.0), gridspec_kw=dict(width_ratios=[1.5, 1]))
    xx = np.linspace(7.8, 11.2, 20)
    ax.errorbar(x_obs, y_obs, xerr=sx, yerr=sy, fmt='o', color='C0', ms=4, elinewidth=0.7, capsize=0)
    ax.plot(xx, TRUE['beta'] + TRUE['alpha'] * (xx - 9.5), '0.4', ls='--', lw=1.4, label=fr"truth: $\alpha$ = {TRUE['alpha']:.2f}")
    ax.plot(xx, th_d[0] + th_d[1] * (xx - 9.5), 'C2', lw=1.6, label=fr'OLS: $\alpha$ = {th_d[1]:.2f} $\pm$ {np.sqrt(cov_d[1, 1]):.2f}')
    ax.plot(xx, beta_odr + alpha_odr * (xx - 9.5), 'C4', lw=1.8, label=fr'ODR: $\alpha$ = {alpha_odr:.2f} $\pm$ {sig_alpha_odr:.2f}')
    ax.set_xlabel(r'$\log_{10}(M_*/M_\odot)$, observed'); ax.set_ylabel(r'$\log_{10}(M_{\rm BH}/M_\odot)$, observed')
    ax.legend(frameon=False, fontsize=9)
    ah.hist(odr_slopes, 40, color='C4', alpha=0.7)
    ah.axvline(TRUE['alpha'], color='0.3', ls='--', lw=1.4, label=f"truth {TRUE['alpha']}")
    ah.axvline(odr_slopes.mean(), color='C4', lw=1.8, label=f'ODR mean {odr_slopes.mean():.2f}')
    ah.set_xlabel(r'ODR slope over 1000 samples'); ah.legend(frameon=False, fontsize=9)
    plt.tight_layout()
    plt.show()
    print(f"ODR on this sample: alpha = {alpha_odr:.2f} +/- {sig_alpha_odr:.2f}, beta = {beta_odr:.2f}; ODR's own reduced chi2 = {odr_out.res_var:.2f}")

## ODR on the demo

* all 40 points, with both error bars
* the ODR fit (left), and ODR on 1000 samples (right)

In [ ]:
# ODR on the same forty points, with both error bars
plot_odr()

* ODR's reduced $\chi^2$ is 1.7, ~3$\sigma$ high for 38 degrees of freedom
* ODR divides each residual by $\sqrt{\sigma_y^2 + \alpha^2\sigma_x^2}$
* a steeper line makes that bigger and $\chi^2$ smaller, so the fit is biased high
* nothing in ODR models the population of true $x$

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**ODR assumes there is no intrinsic scatter. If the model is wrong, the slope is biased high!**

</div>

## The generative model

<div style="display:flex; justify-content:center; align-items:flex-start; gap:0.8em;">
<img src="images/rubin_observatory.jpg" alt="The Vera C. Rubin Observatory on Cerro Pachon" style="max-height:150px;">
<img src="images/euclid.jpg" alt="An image from ESA's Euclid mission" style="max-height:150px;">
<img src="images/roman_telescope.jpg" alt="Illustration of NASA's Nancy Grace Roman Space Telescope" style="max-height:150px;">
</div>

<div style="font-size:0.7em; text-align:center; color:#666;">Rubin: O. Bonin/SLAC (CC BY 4.0). Euclid: ESA/Euclid/Euclid Consortium/NASA, J.-C. Cuillandre, G. Anselmi (CC BY-SA 3.0 IGO). Roman: NASA's Goddard Space Flight Center.</div>

* Charlotte Ward's talk: forward-model the pixels of Rubin, Euclid and Roman together
* let's imagine the forward process for one relation
* each true stellar mass is drawn from a Gaussian population ($\mu$, $\sigma_{\rm pop}$)
* the relation plus intrinsic scatter $\sigma_{\textrm{int}}$ gives the true black-hole mass
* we measure both, with errors
* the true masses are **latent** (from Latin *lateo*, hidden), and the fit integrates over them
* the attenuation depends on $\sigma_{\rm pop}$, and this fit estimates $\sigma_{\rm pop}$ too

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**It's often helpful to explicitly distinguish between observed parameters and latent ones.**

</div>

In [ ]:
# One mock dataset from the generative model, drawn step by step (a "skip" cell)
@one_plot_at_a_time
def plot_mock():
    rng_m = np.random.default_rng(5)
    xt = rng_m.normal(TRUE['mu'], TRUE['sig_pop'], N_demo)
    yt = TRUE['alpha'] * (xt - 9.5) + TRUE['beta'] + rng_m.normal(0, TRUE['sig_int'], N_demo)
    xo = xt + rng_m.normal(0, sx); yo = yt + rng_m.normal(0, sy)
    fig, ax = plt.subplots(figsize=(10, 4.2))
    xx = np.linspace(7.8, 11.2, 20); line = TRUE['alpha'] * (xx - 9.5) + TRUE['beta']
    ax.fill_between(xx, line - TRUE['sig_int'], line + TRUE['sig_int'], color='C3', alpha=0.18, label=r'line $\pm\ \sigma_{\rm int}$ (scatter in true $y$)')
    ax.plot(xx, line, 'k', ls='--', lw=1.4, label='the relation')
    lo, hi = TRUE['mu'] - TRUE['sig_pop'], TRUE['mu'] + TRUE['sig_pop']
    ax.axvline(lo, color='C0', ls=':', lw=1.6); ax.axvline(hi, color='C0', ls=':', lw=1.6, label=r'$\mu \pm \sigma_{\rm pop}$ (spread of true $x$)')
    ax.annotate('', xy=(lo, 3.3), xytext=(hi, 3.3), arrowprops=dict(arrowstyle='<->', color='C0', lw=1.8))
    ax.text(TRUE['mu'], 3.4, r'$2\sigma_{\rm pop}$', color='C0', ha='center', va='bottom', fontsize=11)
    for a, b, c, d in zip(xt, yt, xo, yo):
        ax.annotate('', xy=(c, d), xytext=(a, b), arrowprops=dict(arrowstyle='->', color='0.5', lw=0.7))
    ax.plot(xt, yt, 'o', mfc='white', mec='0.3', ms=5, label='true values (never seen)')
    ax.plot(xo, yo, 'o', color='C0', ms=4, label='observed, after measurement errors')
    ax.set_ylim(3.1, 9.3)
    ax.set_xlabel(r'$\log_{10}(M_*/M_\odot)$'); ax.set_ylabel(r'$\log_{10}(M_{\rm BH}/M_\odot)$')
    ax.legend(frameon=False, fontsize=9, loc='upper left', bbox_to_anchor=(1.01, 1.0))
    plt.tight_layout()
    plt.show()

## A mock dataset from the model

* draw 40 true stellar masses from $\mathcal{N}(\mu, \sigma_{\rm pop})$ - $\sigma_{\rm pop}$ is the spread of true $x$ (blue dotted lines)
* true black-hole mass: the line, plus intrinsic scatter $\sigma_{\textrm{int}}$ in $y$ (red band)
* add measurement errors $\sigma_x$, $\sigma_y$: each arrow goes from true (open) to observed (filled)
* the fit runs this backwards, from the filled points to $\alpha$, $\beta$, $\sigma_{\textrm{int}}$, $\mu$, $\sigma_{\rm pop}$

In [ ]:
# open: true values; filled: what we observe; arrows: the measurement errors
plot_mock()

In [ ]:
# The generative story as a graph: population -> true masses -> observed masses (a "skip" cell)
@one_plot_at_a_time
def plot_pgm():
    from matplotlib.patches import Circle, Rectangle, FancyArrowPatch
    fig, ax = plt.subplots(figsize=(8, 3.2)); ax.set_xlim(0, 10); ax.set_ylim(-0.6, 3.4); ax.axis('off')
    nodes = {'pop': (1.2, 2.6, r'$\mu,\ \sigma_{\rm pop}$', '#c6dbef'), 'rel': (1.2, 0.8, r'$\alpha,\ \beta,\ \sigma_{\rm int}$', '#c6dbef'),
             'xt': (4.0, 2.6, r'$x_{i}^{\rm true}$', 'white'), 'yt': (4.0, 0.8, r'$y_{i}^{\rm true}$', 'white'),
             'xo': (7.2, 2.6, r'$x_i$', '0.6'), 'yo': (7.2, 0.8, r'$y_i$', '0.6')}
    ax.add_patch(Rectangle((2.9, 0.1), 5.4, 3.2, fill=False, lw=1.2, ls='--', color='0.4'))
    ax.text(8.2, 0.2, r'$i = 1 \ldots N$', ha='right', fontsize=10, color='0.4')
    for k, (x, y, lab, col) in nodes.items():
        _r = 0.62 if k in ('pop', 'rel') else 0.42
        ax.add_patch(Circle((x, y), _r, facecolor=col, edgecolor='k', lw=1.2)); ax.text(x, y, lab, ha='center', va='center', fontsize=10 if k == 'rel' else 11)
    for a, b in [('pop', 'xt'), ('rel', 'yt'), ('xt', 'yt'), ('xt', 'xo'), ('yt', 'yo')]:
        (x0, y0), (x1, y1) = nodes[a][:2], nodes[b][:2]
        ax.add_patch(FancyArrowPatch((x0, y0), (x1, y1), arrowstyle='-|>', mutation_scale=14, lw=1.2, color='k', shrinkA=18, shrinkB=18))
    ax.text(5.6, 3.0, r'$+\ \mathcal{N}(0, \sigma_{x,i}^2)$', fontsize=9, ha='center'); ax.text(5.6, 1.2, r'$+\ \mathcal{N}(0, \sigma_{y,i}^2)$', fontsize=9, ha='center')
    ax.text(5.0, -0.35, 'blue: parameters you fit     white: never seen     gray: measured', fontsize=9, color='0.2', ha='center')
    plt.show()

## The same story as a picture

* each arrow means "is drawn from" or "makes", and the dashed box repeats for every galaxy
* gray nodes are measured, blue are what we fit, and white are latent
* (these graphs, probabilistic graphical models, get their own lecture Oct 27/29)

In [ ]:
# the generative model as a graph
plot_pgm()

In [ ]:
# The five-parameter generative fit on the demo sample: bivariate Gaussian per point, curvature error bars (a "skip" cell)
def neg2logL_gen(p, x=x_obs, y=y_obs, sx=sx, sy=sy):
    alpha, beta, log_sint, mu, log_sig_pop = p
    sint2, tau2 = np.exp(2 * log_sint), np.exp(2 * log_sig_pop)
    dx, dy = x - mu, y - (alpha * (mu - 9.5) + beta)
    s11 = tau2 + sx**2
    s22 = alpha**2 * tau2 + sint2 + sy**2
    s12 = alpha * tau2
    det = s11 * s22 - s12**2
    quad = (s22 * dx**2 - 2 * s12 * dx * dy + s11 * dy**2) / det
    return np.sum(np.log(det) + quad)

def numerical_hessian(f, p, h=1e-4):
    p = np.asarray(p, float); n = p.size; H = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            e_i = np.zeros(n); e_j = np.zeros(n); e_i[i] = h; e_j[j] = h
            H[i, j] = (f(p + e_i + e_j) - f(p + e_i - e_j) - f(p - e_i + e_j) + f(p - e_i - e_j)) / (4 * h * h)
    return H

_start = [th_d[1], th_d[0], np.log(0.3), x_obs.mean(), np.log(x_obs.std())]
res_gen = minimize(neg2logL_gen, _start, method='Nelder-Mead', options=dict(xatol=1e-8, fatol=1e-8, maxiter=20000))
p_gen = res_gen.x
cov_gen = 2 * np.linalg.inv(numerical_hessian(neg2logL_gen, p_gen))     # -2lnL curvature -> covariance
alpha_gen, sig_alpha_gen = p_gen[0], np.sqrt(cov_gen[0, 0])
sint_gen = np.exp(p_gen[2])

@one_plot_at_a_time
def plot_generative():
    fig, (ax, ar) = plt.subplots(2, 1, figsize=(7, 5.2), sharex=True, gridspec_kw=dict(height_ratios=[3, 1.1], hspace=0.05))
    xx = np.linspace(7.8, 11.2, 20)
    ax.errorbar(x_obs, y_obs, xerr=sx, yerr=sy, fmt='o', color='C0', ms=4, elinewidth=0.7, capsize=0)
    ax.plot(xx, TRUE['beta'] + TRUE['alpha'] * (xx - 9.5), '0.4', ls='--', lw=1.4, label=fr"truth: $\alpha$ = {TRUE['alpha']:.2f}, $\sigma_\mathrm{{int}}$ = {TRUE['sig_int']:.2f}")
    ax.plot(xx, th_d[0] + th_d[1] * (xx - 9.5), 'C2', lw=1.4, label=fr'OLS: $\alpha$ = {th_d[1]:.2f} $\pm$ {np.sqrt(cov_d[1, 1]):.2f}')
    ax.plot(xx, beta_odr + alpha_odr * (xx - 9.5), 'C4', lw=1.4, label=fr'ODR: $\alpha$ = {alpha_odr:.2f} $\pm$ {sig_alpha_odr:.2f}')
    ax.plot(xx, p_gen[1] + alpha_gen * (xx - 9.5), 'C3', lw=2.2, label=fr'generative: $\alpha$ = {alpha_gen:.2f} $\pm$ {sig_alpha_gen:.2f}, $\sigma_\mathrm{{int}}$ = {sint_gen:.2f}')
    ax.set_ylabel(r'$\log_{10}(M_{\rm BH}/M_\odot)$, observed')
    ax.legend(frameon=False, fontsize=8)
    _pred = p_gen[1] + alpha_gen * (x_obs - 9.5)
    _sig_tot = np.sqrt(sy**2 + sint_gen**2 + alpha_gen**2 * sx**2)     # every source of vertical scatter the model knows about
    ar.axhline(0, color='k', lw=0.6)
    ar.errorbar(x_obs, (y_obs - _pred) / _sig_tot, yerr=1, fmt='o', color='C3', ms=4, elinewidth=0.6)
    ar.set_ylabel(r'(data $-$ fit) / $\sigma_\mathrm{tot}$'); ar.set_xlabel(r'$\log_{10}(M_*/M_\odot)$, observed'); ar.set_ylim(-3.5, 3.5)
    plt.show()
    print(f"generative MLE: alpha = {alpha_gen:.2f} +/- {sig_alpha_gen:.2f} (truth {TRUE['alpha']}, z = {(alpha_gen - TRUE['alpha']) / sig_alpha_gen:+.1f}); "
          f"sigma_int = {sint_gen:.2f} (truth {TRUE['sig_int']}); mu = {p_gen[3]:.2f}, sig_pop = {np.exp(p_gen[4]):.2f} (truth {TRUE['mu']}, {TRUE['sig_pop']})")

## OLS, ODR and the generative fit

* all three on the demo sample, against the dashed truth
* the lower panel shows generative residuals over the total scatter, ~68% within $\pm 1$

In [ ]:
# three estimators on the same forty points, and the line they were all trying to find
plot_generative()

* the generative fit gives 1.13 $\pm$ 0.17, 1.3$\sigma$ from the truth
* it recovers $\sigma_{\textrm{int}}$ (0.39) and $\sigma_{\rm pop}$ (0.63)
* ODR (1.28) is closer on this one sample - the next slide repeats it 1000 times

In [ ]:
# 1000 simulated samples like the demo: OLS, ODR and the generative fit side by side (a "skip" cell)
rng_g = np.random.default_rng(4)
gen_slopes = []
for _ in range(1000):
    xt = rng_g.normal(TRUE['mu'], TRUE['sig_pop'], N_demo)
    yt = TRUE['alpha'] * (xt - 9.5) + TRUE['beta'] + rng_g.normal(0, TRUE['sig_int'], N_demo)
    xo = xt + rng_g.normal(0, sx); yo = yt + rng_g.normal(0, sy)
    A_k = np.vstack([np.ones(N_demo), xo - 9.5]).T
    th_k = np.linalg.lstsq(A_k, yo, rcond=None)[0]
    p_k = minimize(neg2logL_gen, [th_k[1], th_k[0], np.log(0.3), xo.mean(), np.log(xo.std())], args=(xo, yo, sx, sy), method='L-BFGS-B').x
    gen_slopes.append(p_k[0])
gen_slopes = np.array(gen_slopes)

@one_plot_at_a_time
def plot_three_mc():
    fig, ax = plt.subplots(figsize=(8, 3.8))
    bins = np.linspace(0.2, 2.8, 53)
    for sl, col, lab in [(mc_slopes, 'C2', 'OLS'), (odr_slopes, 'C4', 'ODR'), (gen_slopes, 'C3', 'generative')]:
        ax.hist(sl, bins, histtype='step', lw=1.8, color=col, label=f'{lab}: mean {sl.mean():.2f}')
    ax.axvline(TRUE['alpha'], color='k', ls='--', lw=1.4, label=f"truth {TRUE['alpha']}")
    ax.set_xlabel(r'fitted slope $\hat\alpha$ over 1000 samples'); ax.legend(frameon=False, fontsize=9)
    plt.tight_layout()
    plt.show()
    print(f"means: OLS {mc_slopes.mean():.2f}, ODR {odr_slopes.mean():.2f}, generative {gen_slopes.mean():.2f} (median {np.median(gen_slopes):.2f}); truth {TRUE['alpha']}")

## 1000 samples: which estimator is right on average?

* OLS, ODR and the generative fit, each on the same kind of simulated sample, 1000 times

In [ ]:
# three estimators, 1000 samples each, against the truth
plot_three_mc()

* OLS is biased low and ODR is biased high
* the generative fit is centered on the truth
* Kelly 2007 found the same with $10^4$ samples per case (more in the hierarchical Bayes lecture, Oct 27/29)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Model the population of true $x$. The fit estimates $\sigma_{\rm pop}$ and corrects the slope.**

</div>

## Wrapping up regression

* back to Hubble: his distances had the errors, so his $K$ was biased low
* red noise (correlated errors) leaves the slope alone, but OLS error bars are too small
* GLS, with the full $\Sigma$, gets them right
* errors in $x$ bias the OLS slope low by $1 - R_x$, and more data does not help
* ODR is biased high, and the generative model gets the slope right
* the population is really a prior (more on Tuesday), and Thursday is sampling (MCMC)

## Lab error bars vs. the truth

<img src="images/lab_z_summary.png" alt="Three histograms of z, the class's reported answer minus the true value divided by the reported uncertainty, for Lab 01 slope, Lab 02 centroid and Lab 03 transit depth, each with a standard normal curve for comparison" style="display:block;margin:0 auto;max-height:330px">

* ~68% of 1$\sigma$ error bars should contain the truth
* $z$ = (your answer $-$ truth) / your $\sigma$
* if the error bars are right, 95% of $z$ fall within $\pm 2$
* Lab 02 is close, and Labs 01 and 03 each have a few error bars far too small
* in Lab 03 most of you used a block bootstrap (resampling chunks of time), and GLS is the model version
* 4 of 12 Lab 03 answers could not be scored
* fill in the FINAL ANSWER cell, with units!

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Labs 01 and 03 had error bars far too small. Calibrated means ~95% of $|z| < 2$.**

</div>

## Lab 04

<div style="display:flex; justify-content:center; align-items:flex-start; gap:1em;">
<img src="images/dekany2020_ztf_p48_cutaway.jpeg" alt="Cutaway drawing of the Palomar 48-inch Samuel Oschin telescope with the ZTF camera" style="max-height:200px;">
<img src="images/ward2022_fig2_ztf_lightcurve.png" alt="Ward et al. 2022 Figure 2: a ZTF light curve of a variability-selected dwarf AGN candidate" style="max-height:200px;">
<img src="images/wise_pia15809.jpg" alt="Artist's view of the WISE spacecraft over its all-sky infrared map" style="max-height:200px;">
</div>

<div style="font-size:0.7em; text-align:center; color:#666;">Left: ZTF on the Palomar 48-inch, Dekany et al. 2020, PASP 132, 038001 (arXiv:2008.04923), Fig. 1. Middle: a variable dwarf AGN from ZTF, Ward et al. 2022 (arXiv:2110.13098), Fig. 2. Right: WISE, NASA/JPL-Caltech/UCLA (PIA15809).</div>

* your own simulated dwarf-AGN sample, `data/<netid>.csv`
* Part 1 is OLS and the bias, Part 2 is WLS, ODR and the generative fit
* Part 3 is a verification plan, written before you run it
* graded on $z$ with your error bar capped at 3$\times$ your curvature estimate, due Wed Sep 30 by Noon
* **+5**: a page of notes and 2 questions on Charlotte Ward's colloquium, in `submissions/<netid>/`, by Noon Tue Sep 29

## Iben Lecture: David Charbonneau

<div style="display:flex; align-items:flex-start; gap:1.4em; margin-top:0.4em;">
<img src="images/charbonneau2000_fig1_hd209458b.png" alt="Charbonneau et al. 2000 Figure 1: photometry of HD 209458 on two nights in September 1999 showing the transit dip" style="max-height:330px;">
<div>

* colloquium **"Do Rocky Exoplanets Retain Their Atmospheres?"**, Tue Sep 29, 3:30 pm, NCSA Auditorium - **not the usual room or time**
* public lecture **"The Terrestrial Worlds of Other Stars"**, Wed Sep 30, 7 pm, Lincoln Hall Theater
* he co-discovered HD 209458b's transit (left), the first transiting exoplanet, and leads MEarth, the M-dwarf survey in Quiz 2. Go to both

</div>
</div>

<div style="font-size:0.7em; color:#666;">Charbonneau et al. 2000, ApJ 529, L45 (arXiv:astro-ph/9911436), Fig. 1. Listing: calendars.illinois.edu/detail/650.</div>

## Before you go

* **Lab 04** is posted today, due Wed Sep 30 by Noon (fork then PR, as always); every date is in `ASSIGNMENTS.md`
* Ward colloquium notes for the +5 on Lab 04 are due Noon Tue Sep 29
* Tue Sep 29 is the **regression review test**, the whole class, on Zoom (go.illinois.edu/gsn): individual, Copilot allowed, counts as one lab, pull request due Wed Sep 30 by Noon
* Quiz 2 comes back once graded
* Thu Oct 1: Padma Venkatraman guest-lectures in this room on Bayes' theorem, posteriors and sampling them with Markov chain Monte Carlo (MCMC), and Lab 05 posts that day (the dark companion from Day 1)